# AI Infrastructure: Docker, Kubernetes & IaC

A production LLM service is not just application code and a Dockerfile. It is a cluster of compute, a container registry, a secret store, a CI/CD pipeline, and a set of monitoring signals wired together in a reproducible, auditable way. This notebook covers the infrastructure layer end-to-end: containerising the filing analyser service we built in [notebook 10](/courses/llm-eng/10-model-serving.html), deploying it to Kubernetes, provisioning the AWS infrastructure with Terraform, and automating the build-test-push-deploy cycle with GitHub Actions.

Financial services infrastructure has three hard requirements that generic DevOps tutorials ignore: (1) **secret segregation** — API keys, database credentials, and model tokens must never appear in code, images, or logs; (2) **auditability** — every deployment must be traceable to a commit, a PR, and a human approver; (3) **egress control** — containers should not be able to reach arbitrary external endpoints. We address all three in the patterns below.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Kubernetes Deployment

**Why Kubernetes for LLM services.** A single Docker container on a VM works fine for development but provides no self-healing, no horizontal scaling, and no zero-downtime deploys. Kubernetes adds all three. For LLM services specifically, the relevant features are: (1) **horizontal pod autoscaling (HPA)** — automatically add replicas when request latency or CPU exceeds a threshold; (2) **rolling updates** — new image versions are deployed one pod at a time so traffic is never fully dropped; (3) **liveness and readiness probes** — pods that fail their health checks are restarted or removed from load balancing without manual intervention.

<br>

**Resource requests and limits.** LLM service pods should always specify `resources.requests` and `resources.limits`. Requests tell the scheduler how much CPU and memory to reserve; limits cap what the container can actually consume. Without limits a runaway pod (e.g. a memory leak when accumulating conversation history) will OOM its node and evict neighbouring pods. For a gpt-4o-mini proxy, 0.5 CPU and 512 Mi memory per pod is typically sufficient; the bottleneck is network I/O, not compute.

<br>

**Namespace isolation.** Financial services workloads should run in a dedicated Kubernetes namespace (`llm-services`) with a `NetworkPolicy` that restricts egress to the LiteLLM proxy only. This prevents an application bug or compromised pod from exfiltrating data to arbitrary external endpoints.

The complete Kubernetes manifests for the filing analyser service:

```yaml
# k8s/namespace.yaml
apiVersion: v1
kind: Namespace
metadata:
  name: llm-services
---
# k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: filing-analyser
  namespace: llm-services
spec:
  replicas: 2
  selector:
    matchLabels:
      app: filing-analyser
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0          # <1>
  template:
    metadata:
      labels:
        app: filing-analyser
    spec:
      terminationGracePeriodSeconds: 60
      containers:
        - name: app
          image: 123456789.dkr.ecr.us-east-1.amazonaws.com/filing-analyser:latest
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "500m"
              memory: "512Mi"
            limits:
              cpu: "1000m"
              memory: "1Gi"
          lifecycle:
            preStop:
              exec:
                command: ["sleep", "15"]
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            failureThreshold: 3
          env:
            - name: OPENAI_API_KEY
              valueFrom:
                secretKeyRef:
                  name: llm-secrets   # <2>
                  key: openai-api-key
---
# k8s/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: filing-analyser
  namespace: llm-services
spec:
  selector:
    app: filing-analyser
  ports:
    - port: 80
      targetPort: 8000
---
# k8s/hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: filing-analyser-hpa
  namespace: llm-services
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: filing-analyser
  minReplicas: 2
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70
```

1. `maxUnavailable: 0` combined with `maxSurge: 1` means a new pod must be ready before any old pod is terminated — zero-downtime at the cost of briefly running `replicas + 1` pods.
2. The Secret `llm-secrets` is created separately (never in version control) via `kubectl create secret generic llm-secrets --from-env-file=.env.prod`.

## Terraform: AWS ECS + ECR

**Infrastructure as Code.** Manually clicking through the AWS console to create an ECS cluster, a container registry, and the associated IAM roles is fast to do once and impossible to reproduce reliably. Terraform describes infrastructure as declarative HCL files that are version-controlled, code-reviewed, and applied idempotently. The state file tracks what Terraform has already created, so re-applying the same configuration is a no-op if nothing changed.

<br>

**Why ECS over EKS for smaller teams.** Amazon ECS (Elastic Container Service) is a simpler operational target than EKS (Elastic Kubernetes Service) for teams that do not already have Kubernetes expertise. ECS uses the same container concepts (task definitions, services, load balancers) but abstracts away the control plane entirely. ECS Fargate removes node management altogether: tasks run on AWS-managed compute with no EC2 instances to patch or right-size.

<br>

**Secrets Manager integration.** AWS Secrets Manager stores the OpenAI API key as a versioned secret. The ECS task definition references it by ARN — the key is injected as an environment variable at container start time and never written to disk or visible in the task definition JSON stored in version control.

The Terraform configuration for ECR, ECS cluster, and Fargate service:

```hcl
# infra/main.tf
terraform {
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
  backend "s3" {
    bucket = "my-tf-state"
    key    = "filing-analyser/terraform.tfstate"
    region = "us-east-1"
  }
}

variable "openai_api_key" { sensitive = true }   # <1>
variable "image_tag"      { default   = "latest" }

# ── ECR repository ───────────────────────────────────────────────────────────
resource "aws_ecr_repository" "app" {
  name                 = "filing-analyser"
  image_tag_mutability = "MUTABLE"
  image_scanning_configuration { scan_on_push = true }  # <2>
}

# ── Secrets Manager ──────────────────────────────────────────────────────────
resource "aws_secretsmanager_secret" "openai" {
  name = "filing-analyser/openai-api-key"
}
resource "aws_secretsmanager_secret_version" "openai" {
  secret_id     = aws_secretsmanager_secret.openai.id
  secret_string = var.openai_api_key
}

# ── ECS cluster & Fargate service (simplified) ───────────────────────────────
resource "aws_ecs_cluster" "main" {
  name = "llm-services"
}

resource "aws_ecs_task_definition" "app" {
  family                   = "filing-analyser"
  requires_compatibilities = ["FARGATE"]
  network_mode             = "awsvpc"
  cpu                      = 512
  memory                   = 1024
  execution_role_arn       = aws_iam_role.ecs_exec.arn

  container_definitions = jsonencode([{
    name  = "app"
    image = "${aws_ecr_repository.app.repository_url}:${var.image_tag}"
    portMappings = [{ containerPort = 8000 }]
    secrets = [{                              # <3>
      name      = "OPENAI_API_KEY"
      valueFrom = aws_secretsmanager_secret.openai.arn
    }]
    logConfiguration = {
      logDriver = "awslogs"
      options = {
        "awslogs-group"         = "/ecs/filing-analyser"
        "awslogs-region"        = "us-east-1"
        "awslogs-stream-prefix" = "ecs"
      }
    }
  }])
}
```

1. Marking a Terraform variable as `sensitive = true` suppresses its value in plan output and state diffs — it is still stored in the state file, which is why we use an S3 backend with server-side encryption and versioning rather than storing state locally.
2. `scan_on_push = true` runs an AWS Inspector CVE scan on every image pushed to ECR and surfaces findings in the console — a lightweight container security gate that requires no additional tooling.
3. `secrets` in the ECS task definition injects Secrets Manager values as environment variables at container start. The container process reads `os.environ["OPENAI_API_KEY"]` normally; the secret value is never written to the task definition JSON.

**Public HTTPS access requires two more resources**: an Application Load Balancer (ALB) that accepts traffic on port 443 and forwards it to the ECS tasks on port 8000, and an ACM certificate that provides the TLS termination. Terraform manages both. The ALB is the only resource with a public IP — ECS tasks run in private subnets and are never directly reachable from the internet.

The ALB, HTTPS listener, target group, and ACM certificate:

```hcl
# infra/alb.tf

# ── Application Load Balancer ────────────────────────────────────────────────
resource "aws_lb" "app" {
  name               = "filing-analyser-alb"
  internal           = false
  load_balancer_type = "application"
  subnets            = var.public_subnet_ids      # <1>
  security_groups    = [aws_security_group.alb.id]
}

resource "aws_lb_target_group" "app" {
  name        = "filing-analyser-tg"
  port        = 8000
  protocol    = "HTTP"
  vpc_id      = var.vpc_id
  target_type = "ip"                             # <2>

  health_check {
    path                = "/health"
    healthy_threshold   = 2
    unhealthy_threshold = 3
    interval            = 15
  }
}

resource "aws_lb_listener" "https" {
  load_balancer_arn = aws_lb.app.arn
  port              = 443
  protocol          = "HTTPS"
  ssl_policy        = "ELBSecurityPolicy-TLS13-1-2-2021-06"
  certificate_arn   = aws_acm_certificate_validation.app.certificate_arn

  default_action {
    type             = "forward"
    target_group_arn = aws_lb_target_group.app.arn
  }
}

resource "aws_lb_listener" "http_redirect" {     # <3>
  load_balancer_arn = aws_lb.app.arn
  port              = 80
  protocol          = "HTTP"

  default_action {
    type = "redirect"
    redirect {
      port        = "443"
      protocol    = "HTTPS"
      status_code = "HTTP_301"
    }
  }
}

# ── ACM certificate (DNS validation) ─────────────────────────────────────────
resource "aws_acm_certificate" "app" {
  domain_name       = var.domain_name
  validation_method = "DNS"                      # <4>

  lifecycle {
    create_before_destroy = true
  }
}

resource "aws_route53_record" "cert_validation" {
  for_each = {
    for opt in aws_acm_certificate.app.domain_validation_options : opt.domain_name => opt
  }
  zone_id = var.route53_zone_id
  name    = each.value.resource_record_name
  type    = each.value.resource_record_type
  records = [each.value.resource_record_value]
  ttl     = 60
}

resource "aws_acm_certificate_validation" "app" {
  certificate_arn         = aws_acm_certificate.app.arn
  validation_record_fqdns = [for r in aws_route53_record.cert_validation : r.fqdn]
}

output "alb_dns_name" {
  value = aws_lb.app.dns_name                    # <5>
}
```

1. The ALB lives in public subnets (internet-facing); ECS tasks live in private subnets. Only the ALB has an elastic IP — tasks are unreachable directly.
2. `target_type = "ip"` is required for Fargate — there are no EC2 instances to register, only task IP addresses assigned by the VPC.
3. The HTTP-to-HTTPS redirect listener ensures that accidental plain-HTTP requests are silently upgraded rather than failing or exposing data in transit.
4. DNS validation creates a CNAME record in Route 53 that ACM checks to prove domain ownership — it is automated end-to-end with the `aws_route53_record` resource, requiring no manual steps.
5. `alb_dns_name` is the Terraform output you point your CNAME record at, or use directly as the API endpoint in the capstone demo.

## CI/CD with GitHub Actions

**The pipeline contract.** Every merge to `main` should trigger: (1) build and test the Docker image; (2) push the image to ECR tagged with the commit SHA; (3) update the ECS service to pull the new image with a rolling deploy; (4) run a smoke test against the `/health` endpoint of the new deployment. Steps 3 and 4 happen only on `main` — pull requests stop after step 2 to avoid deploying unreviewed code to production.

<br>

**OIDC authentication (no long-lived keys).** The workflow authenticates to AWS using OpenID Connect (OIDC) rather than long-lived `AWS_ACCESS_KEY_ID` secrets stored in GitHub. AWS issues a short-lived token for each workflow run. This eliminates a major attack surface: there are no static credentials to rotate, leak, or revoke.

<br>

**Environment protection rules.** GitHub Environments allow us to require a human approver before the deploy step runs. We define a `production` environment with one required reviewer from the `platform-team`. This creates a mandatory audit trail: every production deployment has an approval event with a timestamp and a named approver, satisfying the auditability requirement.

The GitHub Actions workflow for CI and ECS deployment:

```yaml
# .github/workflows/deploy.yml
name: Build & Deploy

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

env:
  AWS_REGION:  us-east-1
  ECR_REPO:    filing-analyser
  ECS_CLUSTER: llm-services
  ECS_SERVICE: filing-analyser

jobs:
  build:
    runs-on: ubuntu-latest
    permissions:
      id-token: write   # required for OIDC            # <1>
      contents: read
    outputs:
      image: ${{ steps.push.outputs.image }}
    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS credentials (OIDC)
        uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: arn:aws:iam::123456789:role/github-actions-deploy
          aws-region: ${{ env.AWS_REGION }}

      - name: Login to ECR
        id: ecr-login
        uses: aws-actions/amazon-ecr-login@v2

      - name: Build & push image
        id: push
        env:
          REGISTRY: ${{ steps.ecr-login.outputs.registry }}
          SHA:      ${{ github.sha }}
        run: |
          IMAGE="$REGISTRY/$ECR_REPO:$SHA"
          docker build -t "$IMAGE" .
          docker push "$IMAGE"
          echo "image=$IMAGE" >> $GITHUB_OUTPUT          # <2>

  deploy:
    needs: build
    if: github.ref == 'refs/heads/main'                  # <3>
    runs-on: ubuntu-latest
    environment: production
    permissions:
      id-token: write
      contents: read
    steps:
      - name: Configure AWS credentials (OIDC)
        uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: arn:aws:iam::123456789:role/github-actions-deploy
          aws-region: ${{ env.AWS_REGION }}

      - name: Deploy to ECS
        run: |
          aws ecs update-service \
            --cluster $ECS_CLUSTER \
            --service $ECS_SERVICE \
            --force-new-deployment \
            --region $AWS_REGION
          aws ecs wait services-stable \
            --cluster $ECS_CLUSTER \
            --services $ECS_SERVICE
```

1. `id-token: write` is the GitHub permission that allows the workflow to request an OIDC token. Without it, the `configure-aws-credentials` action cannot exchange the token for temporary AWS credentials.
2. Job outputs allow the `image` tag computed in the `build` job to be consumed by the `deploy` job without duplicating the SHA computation.
3. The `deploy` job is gated by both `if: github.ref == 'refs/heads/main'` (branch check) and `environment: production` (human approval gate) — neither condition alone is sufficient.

:::{.callout-caution}
Never store `AWS_ACCESS_KEY_ID` or `AWS_SECRET_ACCESS_KEY` in GitHub Secrets for production deployments. OIDC tokens are short-lived (typically 1 hour) and scoped to a specific repository and branch — a leaked token expires automatically and cannot be used from a different repository.

:::

## Secrets Management

**The secret sprawl problem.** An LLM service in financial services typically needs: an OpenAI (or Azure OpenAI) API key, a Langfuse project key, a Postgres connection string, and possibly a Hugging Face token for open-weight model downloads. Without a systematic approach these end up as hardcoded strings in scripts, as environment variables in unencrypted `.env` files, or duplicated across Kubernetes Secrets that are base64-encoded plaintext. Secret sprawl is the leading cause of credential exposure in cloud environments.

<br>

**The strategy.** We use a three-tier secret hierarchy: (1) AWS Secrets Manager stores the canonical secret values with automatic rotation and audit logging; (2) ECS task definitions reference secrets by ARN, injecting them at runtime with zero copying; (3) for local development, a `.env` file (gitignored) is populated by a `make secrets` target that fetches from Secrets Manager using the developer's personal AWS credentials. The secret value is never stored anywhere other than Secrets Manager and the running process's memory.

<br>

**Rotation.** OpenAI API keys can be rotated from the OpenAI dashboard. We automate this by storing the key in Secrets Manager with `rotation_days = 90` and a Lambda rotation function that calls the OpenAI API to create a new key, updates the secret version, and revokes the old key. ECS tasks pick up the new version on their next restart — HPA-triggered scale-out events effectively act as a rotation trigger.

A helper script for populating a local `.env` file from Secrets Manager:

In [ ]:
import boto3, json, pathlib

SECRET_NAMES = [
    "filing-analyser/openai-api-key",
    "filing-analyser/langfuse-public-key",
    "filing-analyser/langfuse-secret-key",
    "filing-analyser/database-url",
]

ENV_KEY_MAP = {  # Secrets Manager name → .env variable name
    "filing-analyser/openai-api-key":      "OPENAI_API_KEY",
    "filing-analyser/langfuse-public-key": "LANGFUSE_PUBLIC_KEY",
    "filing-analyser/langfuse-secret-key": "LANGFUSE_SECRET_KEY",
    "filing-analyser/database-url":        "DATABASE_URL",
}

def fetch_secrets_to_env(output_path: str = ".env.fetched") -> None:
    sm = boto3.client("secretsmanager", region_name="us-east-1")  # <1>
    lines = ["# Auto-generated by fetch_secrets — do not commit", ""]
    for secret_name in SECRET_NAMES:
        try:
            resp  = sm.get_secret_value(SecretId=secret_name)
            value = resp.get("SecretString", "")
            # Secrets stored as JSON {"value": "..."} or raw string
            try:
                value = json.loads(value).get("value", value)  # <2>
            except json.JSONDecodeError:
                pass
            env_key = ENV_KEY_MAP[secret_name]
            lines.append(f"{env_key}={value}")
        except sm.exceptions.ResourceNotFoundException:
            print(f"WARNING: secret not found: {secret_name}")

    pathlib.Path(output_path).write_text("\n".join(lines) + "\n")
    print(f"wrote {len(SECRET_NAMES)} secrets to {output_path}")

# fetch_secrets_to_env()  # run locally with valid AWS credentials

1. The boto3 client uses the developer's default AWS profile (`~/.aws/credentials`). In CI, the OIDC-vended temporary credentials are set as environment variables by the `configure-aws-credentials` action.
2. Secrets Manager supports both raw string secrets and JSON-structured secrets. We try to parse the value as JSON and extract a `value` field to handle both cases uniformly.

## Monitoring with CloudWatch

**What to monitor.** An LLM service in production needs three categories of signals: (1) **infrastructure metrics** — CPU, memory, request count, P99 latency (from the Application Load Balancer or ECS service metrics); (2) **LLM-specific metrics** — token consumption, cost per request, cache hit rate, model error rate (from our `LLMClient` telemetry); (3) **quality metrics** — hallucination rate, citation accuracy, user thumbs-down rate (from Langfuse, covered in [notebook 06](/courses/llm-eng/06-observability.html)). CloudWatch aggregates categories 1 and 2; Langfuse handles category 3.

<br>

**Custom metrics via CloudWatch Embedded Metric Format (EMF).** Rather than making a synchronous `PutMetricData` API call for every request (which would add latency and cost), we use the Embedded Metric Format: structured JSON written to stdout that the CloudWatch Logs agent extracts and publishes as metrics asynchronously. This requires no SDK call and adds zero latency to the request path.

<br>

**Alarms and dashboards.** We define CloudWatch Alarms for: P99 latency > 8 s (page on-call), error rate > 1% (page on-call), and monthly token cost > $500 (send email). The alarm actions publish to an SNS topic that routes to PagerDuty for the latency/error alarms and to an email subscription for the cost alarm.

Emitting custom metrics in EMF format from the FastAPI request handler:

In [ ]:
import json, time

def emit_emf(namespace: str, metrics: dict, dimensions: dict) -> None:
    """Write an Embedded Metric Format log line to stdout."""
    record = {
        "_aws": {
            "Timestamp": int(time.time() * 1000),
            "CloudWatchMetrics": [{
                "Namespace":  namespace,
                "Dimensions": [list(dimensions.keys())],
                "Metrics":    [{"Name": k, "Unit": v["unit"]} for k, v in metrics.items()],
            }],
        },
        **dimensions,
        **{k: v["value"] for k, v in metrics.items()},  # <1>
    }
    print(json.dumps(record))  # stdout → CloudWatch Logs → EMF extraction

# Example: emit after a filing analysis request
emit_emf(
    namespace="FilingAnalyser",
    metrics={
        "RequestLatencyMs": {"value": 1234.5,  "unit": "Milliseconds"},
        "PromptTokens":     {"value": 512,     "unit": "Count"},
        "CompletionTokens": {"value": 128,     "unit": "Count"},
        "RequestCostUSD":   {"value": 0.00012, "unit": "None"},
    },
    dimensions={
        "Service": "filing-analyser",
        "Model":   "gpt-4o-mini",
        "Env":     "production",
    },
)

1. The EMF record is a flat JSON object containing both the `_aws` metadata structure and the raw metric values at the top level — the CloudWatch Logs agent reads the `_aws` block to know which top-level keys are metrics and which are dimensions.

:::{.callout-note}
EMF requires the CloudWatch Logs agent (`amazon-cloudwatch-agent`) to be configured with the `emf` log format on the ECS task's log group. Without this, the JSON is stored as plain log lines but metrics are never extracted.

:::

## Helm Charts

**Why Helm.** A Kubernetes application is typically 5–10 YAML files (Deployment, Service, HPA, ConfigMap, Secrets, Ingress). Helm packages these into a **chart** — a versioned, parameterised bundle you can install, upgrade, and rollback with a single command. In a financial services context where the same service is deployed across `dev`, `staging`, and `production` with different resource limits and replica counts, Helm values files replace a proliferation of near-identical YAML copies.

<br>

**Chart structure.** The filing analyser chart:

```
charts/filing-analyser/
├── Chart.yaml          # name, version, appVersion
├── values.yaml         # default values (overridden per environment)
├── values.prod.yaml    # production overrides
└── templates/
    ├── deployment.yaml
    ├── service.yaml
    ├── hpa.yaml
    └── _helpers.tpl    # shared template fragments
```

The `Chart.yaml` and parameterised `values.yaml`:

```yaml
# charts/filing-analyser/Chart.yaml
apiVersion: v2
name: filing-analyser
description: LLM-powered SEC filing analysis service
version: 0.1.0        # chart version (bump on chart changes)
appVersion: "1.0.0"   # application version (informational)
```

```yaml
# charts/filing-analyser/values.yaml  (defaults)
replicaCount: 2

image:
  repository: 123456789.dkr.ecr.us-east-1.amazonaws.com/filing-analyser
  tag: latest
  pullPolicy: Always

resources:
  requests:
    cpu: 500m
    memory: 1Gi
  limits:
    cpu: "2"
    memory: 4Gi

autoscaling:
  enabled: true
  minReplicas: 2
  maxReplicas: 10
  targetCPUUtilizationPercentage: 70

env:
  LANGFUSE_HOST: http://langfuse:3000

secretRef: llm-secrets   # K8s Secret name injected as envFrom
```

```yaml
# charts/filing-analyser/values.prod.yaml  (production overrides)
replicaCount: 4
resources:
  requests: {cpu: "1", memory: 2Gi}
  limits:   {cpu: "4", memory: 8Gi}
autoscaling:
  maxReplicas: 20
```

```yaml
# charts/filing-analyser/templates/deployment.yaml  (excerpt)
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "filing-analyser.fullname" . }}
spec:
  replicas: {{ .Values.replicaCount }}
  template:
    spec:
      containers:
      - name: app
        image: {{ .Values.image.repository }}:{{ .Values.image.tag }}  # <1>
        resources: {{ toYaml .Values.resources | nindent 10 }}
        envFrom:
        - secretRef:
            name: {{ .Values.secretRef }}                             # <2>
```


1. `{{ .Values.image.tag }}` is set at deploy time with `--set image.tag=$GITHUB_SHA` — every deployment is tagged with the exact commit SHA that built it, making rollback trivial.
2. `envFrom.secretRef` injects every key in the K8s Secret as an environment variable. The Secret itself is created once with `kubectl create secret generic llm-secrets --from-env-file=.env.prod` and never changes unless credentials rotate.

**Common Helm commands:**

```bash
# Install (first deploy)
helm install filing-analyser ./charts/filing-analyser \
  -f charts/filing-analyser/values.prod.yaml \
  --set image.tag=$GITHUB_SHA                              # <1>

# Upgrade (subsequent deploys)
helm upgrade filing-analyser ./charts/filing-analyser \
  -f charts/filing-analyser/values.prod.yaml \
  --set image.tag=$GITHUB_SHA

# Rollback to previous release
helm rollback filing-analyser 1                          # <2>

# Diff before applying (requires helm-diff plugin)
helm diff upgrade filing-analyser ./charts/filing-analyser \
  --set image.tag=$GITHUB_SHA
```

1. `--set` on the command line overrides a single value without modifying any file — the recommended pattern for injecting dynamic values like image tags in CI/CD.
2. `helm rollback` reinstates the previous release's manifests and re-applies them. Kubernetes performs a rolling update back to the old image — downtime-free revert in under 60 seconds.

:::{.callout-note}
Helm is the standard packaging tool for Kubernetes applications, but it is not required for ECS deployments — ECS uses the Terraform `aws_ecs_task_definition` resource for versioning. The Helm chart above is used when deploying to EKS (Elastic Kubernetes Service) or any self-managed K8s cluster. For the capstone (ECS Fargate), Terraform alone is sufficient.

:::

## Exercises

1. **Write a Terraform module for the LiteLLM proxy.** Create a `modules/litellm-proxy/` directory with `main.tf`, `variables.tf`, and `outputs.tf`. The module should provision an ECS task definition for the LiteLLM Docker image, an ECS service, and a Secrets Manager secret for `LITELLM_MASTER_KEY`. Expose the proxy's internal ALB DNS name as a Terraform output so the filing analyser service can reference it.

2. **Add a cost alarm.** Write a Terraform resource block for a `aws_cloudwatch_metric_alarm` that fires when the monthly sum of `RequestCostUSD` (from the EMF metric above) exceeds $500. The alarm action should publish to an `aws_sns_topic` and subscribe your email address using `aws_sns_topic_subscription`.

3. **Implement a canary deployment.** Extend the GitHub Actions `deploy` job to: (a) deploy the new image to a `filing-analyser-canary` ECS service receiving 10% of traffic via weighted target groups on the ALB; (b) run 20 test requests against the canary; (c) promote to 100% only if the error rate is below 2%; (d) roll back by setting the canary target group weight to 0 if the check fails.

---

$\blacksquare$